# Part 10: MCP vs. Code Reasoning — Interfacing Agents with Databases & IoT Telemetry

In this chapter, we explore a fundamental architectural choice in contemporary agentic AI:

> **How should an LLM interact with databases, live sensors, and structured time series?**

Two competing (yet complementary) paradigms have emerged:
1. **Model Context Protocol (MCP)**: A standardized client-server protocol (championed by Anthropic) where an external server exposes tools, resources, and prompts over JSON-RPC. The agent invokes tools via structured JSON payloads, and the server returns serialized text or JSON.
2. **Code-as-Actions / Code Reasoning (e.g., Hugging Face's `smolagents`)**: The agent reasons by writing executable Python code blocks directly. Instead of streaming raw database dumps through prompt tokens, the agent loads Python modules (`pymongo`, `pandas`, `numpy`), queries the database in-process, transforms data in RAM, and only returns the final computed summary.

---

### Our Real-World Testbed: Linky IoT Telemetry & EV Car Charging (`user.plido.net`)

To compare these paradigms on a concrete, non-trivial problem, we examine a real IoT monitoring deployment:
* **The Server**: An IoT telemetry platform running at `user.plido.net` (database `meteo-data`, collection `measure`).
* **The Source**: An IoT microcontroller (Pycom LoPy) connected via UART to a French smart electricity meter (**Linky TIC**).
* **The Telemetry**: Every few seconds, the meter publishes:
  - `PAPP`: Instantaneous apparent power (in Volt-Amperes / Watts).
  - `BASE`: Cumulative consumption index (in Watt-hours).
  - `IINST`: Instantaneous current (in Amperes).
  - `timestamp` / `date`: ISO 8601 observation timestamp.
* **The User Goal**: 
  > *"Analyze the Linky measurements in MongoDB to detect when my electric car was charging, calculate how long each charging session lasted, determine the energy injected in kWh, and estimate the electricity cost."*

Domestic baseline idle consumption is typically between **200 W and 500 W**. When an electric vehicle (EV) charges via a home wallbox or standard socket (e.g., OpenEVSE), the power leaps to **3,700 W (16 A single-phase)** or up to **6,000 W (25 A)** continuously for several hours.

Let's pit MCP against Code Reasoning to see how each paradigm handles this challenge!


## 1. Setting up our Testbed: The Linky Telemetry Database (`user.plido.net`)

Why use a database like MongoDB to compare MCP and Code Reasoning?
Because time series and database queries produce **structured, multi-record data**. When an agent queries a database for electrical consumption over several days, the result is hundreds or thousands of timestamped records.

How an agent processes these records reveals the deepest architectural differences between:
* **Piping raw documents into prompt tokens** (MCP Tool Calling)
* **Processing documents in-memory via Python reasoning** (`CodeAgent`)

Program 1 attempts to connect directly to the real `user.plido.net` MongoDB database. If the server is unreachable from the outside (port 27017 is typically firewalled for security unless accessed via an SSH tunnel), it automatically falls back to an exact in-memory `mongomock` engine populated with 48 hours of realistic Linky telemetry (including baseline consumption, household appliances, and two distinct EV charging sessions).


In [ ]:
# Program 1: connecting to user.plido.net MongoDB or initializing the Linky testbed

import os
from datetime import datetime, timedelta
from pymongo import MongoClient
import mongomock

# Credentials for user.plido.net
MONGO_HOST = os.environ.get("MONGODB_HOST", "user.plido.net")
MONGO_PORT = int(os.environ.get("MONGODB_PORT", 27017))
MONGO_USER = os.environ.get("MONGODB_USER", "PLIDO")
MONGO_PASS = os.environ.get("MONGODB_PASS", "IMT_2025_imp")
MONGO_URI  = os.environ.get("MONGODB_URI", None)

connected_to_real_server = False

try:
    if MONGO_URI:
        real_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=2000)
    else:
        real_client = MongoClient(
            host=MONGO_HOST,
            port=MONGO_PORT,
            username=MONGO_USER,
            password=MONGO_PASS,
            serverSelectionTimeoutMS=2000
        )
    # Ping to check if host:port is accessible
    real_client.admin.command('ping')
    db = real_client["meteo-data"]
    measure_col = db["measure"]
    n_docs = measure_col.count_documents({})
    print(f"Connected to real MongoDB on {MONGO_HOST}! Collection 'measure' contains {n_docs:,} documents.")
    connected_to_real_server = True
except Exception as e:
    print(f"Direct connection to {MONGO_HOST}:{MONGO_PORT} unavailable ({type(e).__name__}).")
    print("Falling back to local in-memory mongomock testbed mirroring Linky telemetry...")
    
    mock_client = mongomock.MongoClient()
    db = mock_client["meteo-data"]
    measure_col = db["measure"]
    measure_col.drop()
    
    # Generate 48 hours of realistic Linky telemetry (measurements every 2 min = 1,440 points)
    # - Baseline idle power: 250 - 450 W
    # - Daytime appliances: kettle, coffee maker, oven
    # - Session 1 (Night 1): EV charging at 3.7 kW (16A) from 22:30 to 02:15 (3h45)
    # - Session 2 (Night 2): EV fast top-up at 6.0 kW (26A) from 21:00 to 23:10 (2h10)
    base_time = datetime(2026, 3, 14, 0, 0, 0)
    docs = []
    base_index_wh = 45210000  # Index compteur BASE en Wh
    
    for minute in range(0, 2880, 2):
        t = base_time + timedelta(minutes=minute)
        hour = (minute % 1440) / 60.0
        
        # Idle baseline (fridge cycles, standby electronics)
        papp = 320 + (minute % 19) * 6
        iinst = 2
        
        # Day 0: Night EV charging (22:30 to 02:15 next day = minute 1350 to 1575)
        if 1350 <= minute <= 1575:
            papp = 3680 + (minute % 7) * 15
            iinst = 16
        # Day 1: Evening EV fast charge (21:00 to 23:10 = minute 2700 to 2830)
        elif 2700 <= minute <= 2830:
            papp = 5950 + (minute % 9) * 20
            iinst = 26
        # Appliance spikes: morning coffee & kettle (07:15 to 07:35)
        elif 7.25 <= hour <= 7.6:
            papp += 1800
            iinst += 8
        # Appliance spikes: evening cooking (19:30 to 20:15)
        elif 19.5 <= hour <= 20.25:
            papp += 2200
            iinst += 10
            
        base_index_wh += int(papp * (2 / 60.0))
        docs.append({
            "timestamp": t.isoformat(),
            "date": t,
            "PAPP": papp,
            "IINST": iinst,
            "BASE": base_index_wh,
            "Location": "Linky-Maison",
            "OPTARIF": "HC..",
            "ISOUSC": 45
        })
        
    measure_col.insert_many(docs)
    print(f"Testbed initialized: {measure_col.count_documents({})} Linky telemetry frames created in 'meteo-data.measure'.")
    print(f"Time range: {docs[0]['timestamp']} -> {docs[-1]['timestamp']}")


## 2. Paradigm A: The MCP Way (Tool Calling over JSON-RPC)

In the Model Context Protocol (MCP) vision:
1. The database is shielded behind a dedicated **MCP Server**.
2. The server exposes a set of pre-defined tools (decorated with `@mcp.tool()`) and resources over standard I/O (`stdio`) or HTTP.
3. The LLM acts as an **orchestrator**: it sends a JSON tool-call request, the client forwards it to the server, and the server returns a JSON/text result that is injected into the LLM's conversation history.

Let's build a dedicated FastMCP server script `data/linky_mcp_server.py`. It provides two tools:
* `get_meter_summary()`: returns the meter configuration and latest reading.
* `detect_ev_charging_sessions(min_power_watts, min_duration_minutes)`: a pre-baked detection algorithm running on the server.


In [ ]:
# Program 2: creating an MCP server that exposes Linky operations

import os

DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "data"))
os.makedirs(DATA_DIR, exist_ok=True)
MCP_SERVER_PATH = os.path.join(DATA_DIR, "linky_mcp_server.py")

server_code = """\"\"\"An MCP server providing controlled access to Linky electricity telemetry in MongoDB.\"\"\"

import json
from datetime import datetime, timedelta
import mongomock
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Linky-Telemetry-Server")

# Recreate mock database inside the server process
client = mongomock.MongoClient()
db = client["meteo-data"]
col = db["measure"]

# Populate the same Linky testbed dataset
col.drop()
base_time = datetime(2026, 3, 14, 0, 0, 0)
docs = []
base_index_wh = 45210000

for minute in range(0, 2880, 2):
    t = base_time + timedelta(minutes=minute)
    hour = (minute % 1440) / 60.0
    papp = 320 + (minute % 19) * 6
    iinst = 2
    if 1350 <= minute <= 1575:
        papp = 3680 + (minute % 7) * 15
        iinst = 16
    elif 2700 <= minute <= 2830:
        papp = 5950 + (minute % 9) * 20
        iinst = 26
    elif 7.25 <= hour <= 7.6:
        papp += 1800
        iinst += 8
    elif 19.5 <= hour <= 20.25:
        papp += 2200
        iinst += 10
    base_index_wh += int(papp * (2 / 60.0))
    docs.append({
        "timestamp": t.isoformat(),
        "date": t,
        "PAPP": papp,
        "IINST": iinst,
        "BASE": base_index_wh,
        "Location": "Linky-Maison",
        "OPTARIF": "HC..",
        "ISOUSC": 45
    })
col.insert_many(docs)

@mcp.tool()
def get_meter_summary() -> str:
    \"\"\"Return the Linky meter status, subscribed power, and latest reading.\"\"\"
    last_doc = col.find_one(sort=[("date", -1)])
    if not last_doc:
        return "No telemetry available in database."
    return json.dumps({
        "location": last_doc.get("Location"),
        "optarif": last_doc.get("OPTARIF"),
        "isousc_kva": last_doc.get("ISOUSC"),
        "last_papp_watts": last_doc.get("PAPP"),
        "total_index_kwh": last_doc.get("BASE") / 1000.0,
        "last_timestamp": last_doc.get("timestamp")
    }, indent=2)

@mcp.tool()
def detect_ev_charging_sessions(min_power_watts: int = 3000, min_duration_minutes: int = 20) -> str:
    \"\"\"Detect sustained high-power charging sessions characteristic of an electric vehicle.
    
    Args:
        min_power_watts: Threshold power in Watts (typical EV: >= 3000 W).
        min_duration_minutes: Minimum duration to filter out short culinary spikes.
    \"\"\"
    high_power_docs = list(col.find({"PAPP": {"$gte": min_power_watts}}).sort("date", 1))
    sessions = []
    current = []
    
    for doc in high_power_docs:
        if not current:
            current.append(doc)
        else:
            prev_t = current[-1]["date"]
            if (doc["date"] - prev_t).total_seconds() <= 300:
                current.append(doc)
            else:
                sessions.append(current)
                current = [doc]
    if current:
        sessions.append(current)
        
    results = []
    for s in sessions:
        start_t = s[0]["date"]
        end_t = s[-1]["date"]
        duration_m = (end_t - start_t).total_seconds() / 60.0
        if duration_m >= min_duration_minutes:
            kwh = (s[-1]["BASE"] - s[0]["BASE"]) / 1000.0
            avg_p = sum(d["PAPP"] for d in s) / len(s)
            results.append({
                "start": start_t.isoformat(),
                "end": end_t.isoformat(),
                "duration_minutes": round(duration_m, 1),
                "avg_power_watts": round(avg_p, 0),
                "energy_kwh": round(kwh, 2),
                "estimated_cost_euros": round(kwh * 0.2516, 2)
            })
            
    return json.dumps(results, indent=2)

if __name__ == "__main__":
    mcp.run()
"""

with open(MCP_SERVER_PATH, "w") as f:
    f.write(server_code)

print(f"MCP server script written to: {MCP_SERVER_PATH}")


Now let's launch an agent that connects to our Linky MCP server and ask it our core question:

> *"Analyze my Linky telemetry. When did my electric vehicle charge, how long did each session last, how many kWh were consumed, and what did it cost at 0.25 €/kWh?"*


In [ ]:
# Program 3: running an agent against the Linky MCP server

import sys
from contextlib import AsyncExitStack
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display

# Configure model (using local Ollama qwen3:8b, or switch to Rennes/ILaaS/OpenAI)
ollama_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
model = OpenAIChatCompletionsModel(model="qwen3:8b", openai_client=ollama_client)

async def ask_mcp_agent(query: str):
    async with AsyncExitStack() as stack:
        mcp_params = {
            "command": sys.executable,
            "args": [MCP_SERVER_PATH]
        }
        server = await stack.enter_async_context(MCPServerStdio(mcp_params))
        
        agent = Agent(
            name="Linky Energy Advisor",
            instructions=(
                "You are an expert energy advisor analyzing smart meter telemetry from a Linky meter. "
                "Use the available MCP tools to examine power measurements and report EV charging sessions "
                "clearly in a Markdown table with start time, duration, kWh delivered, and cost."
            ),
            model=model,
            mcp_servers=[server]
        )
        
        print(f"Querying agent: '{query}'\n")
        result = await Runner.run(agent, query)
        return result.final_output

# In a Jupyter notebook, run via asyncio:
import nest_asyncio
nest_asyncio.apply()
import asyncio

try:
    response = asyncio.run(ask_mcp_agent(
        "Check my Linky database: did my electric car charge recently? "
        "List each session with timestamps, duration, kWh consumed, and total cost."
    ))
    display(Markdown(response))
except Exception as e:
    print(f"Agent execution encountered: {e}")


### Look closely at what just happened under the hood:

```mermaid
sequenceDiagram
    participant LLM as Agent (LLM)
    participant Client as MCP Client
    participant Server as FastMCP Server
    participant DB as MongoDB (meteo-data)

    LLM->>Client: JSON Tool Call: detect_ev_charging_sessions(min_power_watts=3000)
    Client->>Server: JSON-RPC request over stdio
    Server->>DB: col.find({"PAPP": {"$gte": 3000}})
    DB-->>Server: Raw MongoDB documents
    Server-->>Client: Serialized JSON array of sessions
    Client-->>LLM: Injects JSON text into prompt context
    LLM->>LLM: Formats answer into Markdown
```

#### The Strengths of MCP:
1. **Safety & Encapsulation**: The LLM never writes raw database queries or executable scripts. It only calls sanctioned functions exposed by the server.
2. **Standardization**: Any MCP client (Claude Desktop, Antigravity, custom agent) can query `data/linky_mcp_server.py` using the exact same JSON-RPC protocol.

#### The Achilles' Heel of MCP:
1. **The "Pre-Baked Tool" Paradox**: Why did this succeed cleanly? Because the server author **already wrote the exact Python detection logic** (`detect_ev_charging_sessions`) inside the MCP server beforehand!
2. **The Token Explosion Trap**: What if the user asked a slightly different question:
   > *"What was my average power consumption between 18:00 and 22:00 over the past 2 weeks, excluding weekends?"*
   If the server only has generic tools like `get_all_measurements()`, the server must serialize **thousands of Linky records into JSON text** and dump them into the LLM's prompt window.
   - 2 weeks of Linky telemetry @ 1 measure every 10 seconds = **120,960 JSON objects**.
   - Even at 50 tokens per object, that is **6,048,000 tokens** — instantly crashing context limits and costing hundreds of euros!


## 3. Paradigm B: Code-as-Actions (`CodeAgent` with Python Reasoning)

Now let's switch to the alternative paradigm: **Code Reasoning**, as implemented by Hugging Face's `smolagents`.

In this approach, the agent does **NOT** emit JSON tool requests. It writes **executable Python code blocks**.

Instead of streaming thousands of database documents through the LLM prompt:
1. The agent writes Python code that queries `measure_col` (via `pymongo` or high-level query helpers).
2. The query results stay **in Python variables in RAM**.
3. The LLM uses Python loops, list comprehensions, and standard libraries (`datetime`, `math`, `pandas`, `numpy`) to segment the time series, calculate duration, and integrate $\Delta \text{BASE}$ (or $\int P \, dt$).
4. Only the **final printed summary table** is returned to the user!


In [ ]:
# Program 4: solving the car charging detection with a smolagents CodeAgent

from smolagents import CodeAgent, OpenAIServerModel, tool
from datetime import datetime

# 1. Provide a focused query tool to fetch telemetry from MongoDB
@tool
def query_linky_telemetry(min_power: int = 0) -> list:
    """Fetch Linky power records from the MongoDB 'meteo-data.measure' collection.
    
    Args:
        min_power: Minimum apparent power (PAPP in Watts) to retrieve. 
                   Pass 3000 to isolate high-power EV charging events.
    """
    cursor = measure_col.find(
        {"PAPP": {"$gte": min_power}},
        {"_id": 0, "timestamp": 1, "PAPP": 1, "IINST": 1, "BASE": 1}
    ).sort("date", 1)
    return list(cursor)

# 2. Configure model for smolagents
smol_model = OpenAIServerModel(
    model_id="qwen3:8b",
    api_base="http://localhost:11434/v1",
    api_key="ollama"
)

# 3. Create the CodeAgent with Python reasoning capabilities
code_agent = CodeAgent(
    tools=[query_linky_telemetry],
    model=smol_model,
    additional_authorized_imports=["datetime", "math", "statistics", "pandas", "json"]
)

task = (
    "Query the Linky telemetry from MongoDB for power >= 3000 W using query_linky_telemetry(min_power=3000). "
    "In Python, group contiguous records (where time gap between successive points is <= 5 minutes) "
    "into distinct electric vehicle charging sessions. "
    "For each session: "
    "1) Identify start and end timestamps. "
    "2) Calculate duration in minutes. "
    "3) Compute average power in Watts. "
    "4) Compute total energy consumed in kWh (by dividing the difference in BASE index by 1000). "
    "5) Calculate the cost at 0.25 €/kWh. "
    "Print a clean summary table."
)

print(f"Running CodeAgent on task:\n{task}\n")
try:
    code_result = code_agent.run(task)
    print("\n=== Agent Output ===")
    print(code_result)
except Exception as e:
    print(f"CodeAgent execution: {e}")


### Look at the execution trace of the CodeAgent:

```mermaid
flowchart TD
    A["User Task: 'Find when my EV charged'"] --> B["LLM generates Python code"]
    B --> C["Local Python Sandbox executes code"]
    C --> D["query_linky_telemetry(min_power=3000)"]
    D --> E["RAM: 200 Linky records loaded in memory"]
    E --> F["Python loop segments records into contiguous sessions (gap <= 5 min)"]
    F --> G["Python computes: duration, kWh from delta BASE, cost"]
    G --> H["Print Markdown Table (150 tokens)"]
    H --> I["User receives final answer"]
```

Notice the crucial difference:
* **The raw records NEVER entered the LLM's prompt context!**
* The LLM didn't have to "read" 200 documents line by line. It simply wrote the standard Python logic to segment them, execute the arithmetic, and display the result.
* The entire data pipeline ran at native C/Python speed in RAM.


## 4. Head-to-Head Comparison: MCP vs. Code-as-Actions

| Architectural Dimension | Paradigm A: MCP (JSON-RPC Protocol) | Paradigm B: Code-as-Actions (`CodeAgent`) |
|---|---|---|
| **Action representation** | Structured JSON tool call (`{"name": "...", "args": {...}}`) | Executable Python code block (````python ... ````) |
| **Data pipeline** | Tool results are serialized to text and fed into LLM prompt | Data stays in Python RAM variables; only summary returns |
| **Complex logic & loops** | Requires $N$ separate LLM round-trips for $N$ steps | Solved in 1 turn via Python `for` loops and comprehensions |
| **Ad-hoc aggregation** | Fails unless server author already implemented that exact tool | LLM writes arbitrary filters, joins, groupings, and math |
| **Context Window scaling** | $\mathcal{O}(N)$ with dataset size (explodes on large queries) | $\mathcal{O}(1)$ — constant prompt size regardless of data size |
| **Security boundary** | Strong: server controls strictly what code can run | Requires a sandboxed Python execution environment |
| **Cross-tool composition** | Tool A result $\to$ LLM text $\to$ Tool B call (verbose) | Tool B(Tool A()) chained natively in Python code |


## 5. The Best of Both Worlds: A CodeAgent calling MCP Servers

Are these two paradigms mutually exclusive? **No!**

The industry consensus is converging toward a **hybrid architecture**:
* **Infrastructure Layer**: Databases, internal APIs, file systems, and IoT brokers are exposed as standard, secured **MCP Servers**.
* **Reasoning Layer**: The agent is a **CodeAgent**. Instead of calling tools via rigid JSON round-trips, the CodeAgent dynamically discovers MCP tools and imports them directly as **callable Python functions**!

In `smolagents`, this is natively supported via `MCPClient`:

```python
from smolagents import CodeAgent, MCPClient
from mcp import StdioServerParameters

# 1. Connect to our Linky MCP server
mcp_client = MCPClient(
    StdioServerParameters(
        command="python",
        args=["data/linky_mcp_server.py"]
    )
)

# 2. Give the MCP tools directly to the CodeAgent!
agent = CodeAgent(
    tools=mcp_client.get_tools(),
    model=smol_model
)

# The agent can now write Python code that calls mcp_tools inside loops!
agent.run("Use get_meter_summary() to inspect the meter, then compute the remaining capacity under our 45A subscription.")
```


In [ ]:
# Program 5: demonstrating direct Python data processing on Linky telemetry

# Here is how a CodeAgent reasons natively with pymongo and pandas:
import pandas as pd

docs = list(measure_col.find({"PAPP": {"$gte": 3000}}, {"_id": 0, "date": 1, "PAPP": 1, "IINST": 1, "BASE": 1}))
df = pd.DataFrame(docs)

if not df.empty:
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")
    
    # Identify gaps > 5 min to split charging sessions
    df["gap"] = (df["date"].diff() > pd.Timedelta(minutes=5)).cumsum()
    
    summary = []
    for session_id, group in df.groupby("gap"):
        start_t = group["date"].iloc[0]
        end_t = group["date"].iloc[-1]
        duration_m = (end_t - start_t).total_seconds() / 60.0
        kwh = (group["BASE"].iloc[-1] - group["BASE"].iloc[0]) / 1000.0
        avg_w = group["PAPP"].mean()
        summary.append({
            "Session": session_id + 1,
            "Start": start_t.strftime("%Y-%m-%d %H:%M"),
            "End": end_t.strftime("%H:%M"),
            "Duration (min)": f"{duration_m:.0f}",
            "Avg Power (W)": f"{avg_w:.0f}",
            "Energy (kWh)": f"{kwh:.2f}",
            "Cost (€)": f"{kwh * 0.2516:.2f} €"
        })
        
    df_summary = pd.DataFrame(summary)
    print("=== EV Charging Sessions Detected via Python Reasoning ===")
    print(df_summary.to_string(index=False))
else:
    print("No charging sessions found above 3000 W.")


## 6. Student Exercises

### Exercise 1: Expanding the FastMCP Server (Tool Calling)

**Goal**: In `data/linky_mcp_server.py`, add a new tool to compute the split between Heures Creuses (Off-Peak) and Heures Pleines (Peak hours).

**Context**: In France, Linky contracts often feature two tariff periods:
* **Heures Creuses (HC)**: typically night hours (22:30 to 06:30) with cheaper electricity (~0.18 €/kWh).
* **Heures Pleines (HP)**: daytime hours (06:30 to 22:30) with regular pricing (~0.27 €/kWh).

**Task**:
1. Implement a function `get_tariff_breakdown(start_date, end_date)` directly on `measure_col`.
2. Segment the consumption into HC (hours where `hour >= 22.5` or `hour < 6.5`) vs. HP (daytime).
3. Compute the total kWh and cost for each period.


In [ ]:
# Exercise 1: Implement get_tariff_breakdown directly on Linky telemetry

def get_tariff_breakdown() -> dict:
    """Compute total kWh consumed in Heures Creuses vs. Heures Pleines."""
    # TODO: Iterate over measure_col documents and accumulate kWh
    hc_wh = 0
    hp_wh = 0
    
    docs = list(measure_col.find().sort("date", 1))
    for i in range(1, len(docs)):
        prev_doc = docs[i-1]
        curr_doc = docs[i]
        
        # Energy delta between two successive points in Wh
        delta_wh = curr_doc["BASE"] - prev_doc["BASE"]
        hour = curr_doc["date"].hour + curr_doc["date"].minute / 60.0
        
        # Heures Creuses: 22h30 to 06h30
        if hour >= 22.5 or hour < 6.5:
            hc_wh += delta_wh
        else:
            hp_wh += delta_wh
            
    return {
        "Heures Creuses (kWh)": round(hc_wh / 1000.0, 2),
        "Heures Pleines (kWh)": round(hp_wh / 1000.0, 2),
        "Cost HC (0.18€/kWh)": round((hc_wh / 1000.0) * 0.18, 2),
        "Cost HP (0.27€/kWh)": round((hp_wh / 1000.0) * 0.27, 2),
        "Total Cost (€)": round((hc_wh / 1000.0) * 0.18 + (hp_wh / 1000.0) * 0.27, 2)
    }

print("Tariff Breakdown:")
print(get_tariff_breakdown())


<details>
<summary><b>Click here to view the discussion & solution for Exercise 1</b></summary>

Notice how, in the MCP paradigm, every new question from a user requires a human developer to anticipate the need, write a new Python function with `@mcp.tool()`, restart the MCP server, and update tool schemas.

If the user tomorrow asks: *"What is my consumption exclusively on Sunday afternoons?"*, an MCP agent without a pre-baked tool is completely stuck!

In contrast, a `CodeAgent` can answer any arbitrary temporal slice on the fly.
</details>


### Exercise 2: Non-Intrusive Load Monitoring (NILM) with `CodeAgent`

**Goal**: Separate EV charging sessions from high-power culinary appliances (oven, kettle, coffee maker).

**Task**:
Using `code_agent` (Program 4), ask the following prompt:
> *"Perform Non-Intrusive Load Monitoring (NILM) on the Linky telemetry. Identify two distinct categories of high-power events: 1) Short appliance spikes (power between 1,500 W and 2,500 W lasting under 45 minutes, e.g. kettle/oven), and 2) Vehicle charging sessions (power >= 3,000 W lasting over 1 hour). Print a comparative summary of both categories."*


In [ ]:
# Exercise 2: Run the NILM appliance classification with CodeAgent

nilm_task = (
    "Using query_linky_telemetry(min_power=1500), perform NILM event classification: "
    "1) Group contiguous power periods (gap <= 5 min). "
    "2) Classify each event: "
    "   - 'Domestic Appliance' if average power is between 1500W and 2500W AND duration < 45 minutes. "
    "   - 'EV Charging' if average power >= 3000W AND duration >= 60 minutes. "
    "3) Display a table with category, start time, duration, and energy in kWh."
)

print(f"Running NILM query with CodeAgent...\n")
try:
    nilm_result = code_agent.run(nilm_task)
    print("\n=== NILM Output ===")
    print(nilm_result)
except Exception as e:
    print(f"NILM execution error: {e}")


<details>
<summary><b>Click here to view the discussion & solution for Exercise 2</b></summary>

Here is the exact Python logic that the `CodeAgent` executes in RAM:

```python
docs = query_linky_telemetry(min_power=1500)
# Group into events by timestamp gap <= 300s
events = []
current = []
for d in docs:
    if not current or (d['date'] - current[-1]['date']).total_seconds() <= 300:
        current.append(d)
    else:
        events.append(current)
        current = [d]
if current:
    events.append(current)

for ev in events:
    duration_min = (ev[-1]['date'] - ev[0]['date']).total_seconds() / 60.0
    avg_power = sum(x['PAPP'] for x in ev) / len(ev)
    kwh = (ev[-1]['BASE'] - ev[0]['BASE']) / 1000.0
    
    if 1500 <= avg_power <= 2500 and duration_min < 45:
        category = "Domestic Appliance (Oven/Kettle)"
    elif avg_power >= 3000 and duration_min >= 60:
        category = "EV Charging"
    else:
        category = "Other / Transient"
    print(f"[{category}] Start: {ev[0]['timestamp']} | Duration: {duration_min:.0f}m | Energy: {kwh:.2f} kWh")
```

The agent executes complex signal processing algorithms without transferring raw data back and forth to the LLM.
</details>


### Exercise 3: The Context Window & Token Explosion Benchmark

Let's calculate the economic and technical consequences of both paradigms on real Linky telemetry.

**Data scale**:
* A Linky smart meter transmits a TIC frame approximately **every 10 seconds**.
* That represents **6 frames per minute = 360 frames/hour = 8,640 frames/day**.
* Over a typical monthly billing cycle (30 days), the database collects **259,200 telemetry documents**.
* A compact JSON representation of each measurement `{ "timestamp": "...", "PAPP": 3680, "IINST": 16, "BASE": 45210000 }` requires approximately **50 tokens**.

**Questions to compute**:
1. If an MCP server returns the raw 30-day telemetry to the agent via `query_measurements()`, how many tokens would be sent in the prompt?
2. At \$1.50 per 1M input tokens (e.g. Gemini 2.5 Flash / GPT-4o-mini), how much would that single query cost?
3. How many prompt tokens does the `CodeAgent` approach consume for the exact same 30-day dataset?


<details>
<summary><b>Click here to view the calculation & solution for Exercise 3</b></summary>

#### Calculations:

1. **Total Tokens for MCP Raw Data**:
   $$259{,}200 \text{ documents} \times 50 \text{ tokens/doc} = 12{,}960{,}000 \text{ tokens}$$
   *(12.96 million tokens! This exceeds the maximum context window of nearly every model in existence).*

2. **Cost for a single query via MCP**:
   $$\frac{12{,}960{,}000}{1{,}000{,}000} \times 1.50\text{ \$} = 19.44\text{ \$ per query!}$$
   If an IoT dashboard runs this query hourly or daily for 100 households, the API bill would exceed **\$58,000 / month**.

3. **Tokens consumed by the `CodeAgent` approach**:
   The prompt sent to the LLM is only the instruction (~150 tokens) + tool signature (~100 tokens) = **~250 tokens**.
   The 259,200 records are filtered and aggregated inside Python's RAM by the Python interpreter.
   The returned output is simply the final summary table (~200 tokens).
   Total cost:
   $$\frac{450}{1{,}000{,}000} \times 1.50\text{ \$} = 0.000675\text{ \$}$$
   **The CodeAgent approach is nearly 30,000 times cheaper and fits comfortably within any LLM context window!**
</details>


## Key Takeaways

1. **MCP (Model Context Protocol)** is an outstanding standard for **infrastructure and tool federation**:
   - It provides uniform, secure discovery of external services (APIs, databases, filesystems).
   - It avoids custom SDK lock-in for tool servers.

2. **Tool-Calling over JSON-RPC** breaks down when dealing with **dense data and time series**:
   - Streaming raw database records into prompt tokens causes catastrophic context bloat and prohibitive costs.
   - It forces developers to pre-bake every single analytical metric as a custom server tool.

3. **Code-as-Actions (`CodeAgent`)** provides a vastly superior computational model for **data-heavy tasks**:
   - Code reasoning manipulates data in RAM using Python's rich ecosystem (`pandas`, `numpy`, `math`).
   - The LLM writes the logic; Python executes the loop.
   - Complexity stays $\mathcal{O}(1)$ relative to the database size.

4. **The Modern Synthesis**:
   - Build **MCP servers** to secure and expose your data infrastructure.
   - Use a **CodeAgent** on the client side to import those MCP tools as Python functions, executing loops and analytical aggregations inside the local code sandbox!
